# CycleGAN + CBAM — Colab training notebook

Trains the OCT image-enhancement model on a Colab GPU.

**Before running:** make sure your dataset is uploaded to Drive at:
```
My Drive/medical-gan-oct/data/processed/domain_A/
My Drive/medical-gan-oct/data/processed/domain_B/
```

**Runtime → Change runtime type → GPU** (T4 is fine; L4/A100 if you have Pro).

Run the cells top to bottom. Total wall time on T4: ~10 minutes for 200 epochs.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo and install

We `git clone` instead of editing in Colab so the source of truth stays on GitHub.

In [ ]:
%cd /content
![ -d medical-gan-oct ] || git clone https://github.com/BLHmarwane/medical-gan-oct.git
%cd medical-gan-oct
!git pull
!pip install -q -r requirements.txt
!pip install -q -e .

## 4. Link Drive folders into the workspace

Symlinks make the Drive paths look like local paths — no code changes needed.
Checkpoints and sample PNGs written by training will land directly in your Drive.

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/medical-gan-oct')
REPO_ROOT  = Path('/content/medical-gan-oct')

# Ensure target directories exist on Drive
(DRIVE_ROOT / 'data/processed/domain_A').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'data/processed/domain_B').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'logs/samples').mkdir(parents=True, exist_ok=True)

def symlink(src: Path, dst: Path) -> None:
    if dst.is_symlink() or dst.exists():
        if dst.is_symlink():
            dst.unlink()
        else:
            import shutil; shutil.rmtree(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.symlink_to(src)

symlink(DRIVE_ROOT / 'data/processed', REPO_ROOT / 'data/processed')
symlink(DRIVE_ROOT / 'checkpoints',    REPO_ROOT / 'checkpoints')
symlink(DRIVE_ROOT / 'logs',           REPO_ROOT / 'logs')

print('Symlinks ready:')
!ls -la /content/medical-gan-oct/data /content/medical-gan-oct/checkpoints /content/medical-gan-oct/logs

## 5. Sanity-check the dataset

In [ ]:
import os
a = os.listdir('/content/medical-gan-oct/data/processed/domain_A')
b = os.listdir('/content/medical-gan-oct/data/processed/domain_B')
print(f'domain_A: {len(a)} files')
print(f'domain_B: {len(b)} files')
assert len(a) > 0 and len(b) > 0, 'Upload your data to Drive before training.'

## 6. Train

On T4: ~3 seconds per epoch → ~10 minutes for 200 epochs.
Sample PNGs and checkpoints stream to Drive as they're written.

In [ ]:
!python scripts/train.py --config configs/cyclegan_cbam_colab.yaml

## 7. Preview the latest sample grid

In [ ]:
from pathlib import Path
from IPython.display import Image, display

samples = sorted(Path('/content/medical-gan-oct/logs/samples').glob('epoch_*.png'))
if samples:
    print(f'{len(samples)} sample grids saved. Showing the latest:')
    display(Image(filename=str(samples[-1])))
else:
    print('No samples yet — training may still be starting.')